# Gold Serving Table: Current Line Status

Build the latest operational status for every current Tube line.

**Source:** `workspace.urbanpulse_gold.fact_line_status`

**Dimension:** `workspace.urbanpulse_gold.dim_line`

**Target:** `workspace.urbanpulse_gold.current_line_status`

**Grain:** One row per current Tube line.

This table is designed for Databricks Apps, SQL dashboards, and downstream application consumption.

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.current_line_status import (
    build_current_line_status,
)

from urbanpulse.quality.current_line_status import (
    invalid_current_line_status,
)

In [0]:
FACT_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "fact_line_status"
)

DIM_LINE_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "dim_line"
)

TARGET_TABLE = (
    "workspace."
    "urbanpulse_gold."
    "current_line_status"
)

In [0]:
fact_df = spark.table(
    FACT_TABLE
)

dim_line_df = spark.table(
    DIM_LINE_TABLE
)

current_line_count = (
    dim_line_df
    .filter(F.col("is_current"))
    .count()
)

print(
    f"Current dimension lines: "
    f"{current_line_count}"
)

In [0]:
current_status_df = (
    build_current_line_status(
        fact_df=fact_df,
        dim_line_df=dim_line_df,
    )
)

serving_count = (
    current_status_df.count()
)

print(
    f"Serving rows: "
    f"{serving_count}"
)

display(
    current_status_df
    .orderBy("line_name")
)

In [0]:
if serving_count != current_line_count:
    raise ValueError(
        "Current line status does not contain "
        "exactly one row per current Tube line."
    )

print(
    "Current line coverage validation passed."
)

In [0]:
duplicate_lines_df = (
    current_status_df
    .groupBy("line_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_lines_df.count() > 0:
    display(
        duplicate_lines_df
    )

    raise ValueError(
        "Duplicate lines detected in "
        "current_line_status."
    )

print(
    "Serving grain validation passed."
)

In [0]:
invalid_df = (
    invalid_current_line_status(
        current_status_df
    )
)

invalid_count = (
    invalid_df.count()
)

print(
    f"Invalid rows: {invalid_count}"
)

if invalid_count > 0:
    display(invalid_df)

    raise ValueError(
        f"{invalid_count} invalid current "
        "line status rows detected."
    )

print(
    "Serving quality checks passed."
)

In [0]:
invalid_flags_df = (
    current_status_df
    .filter(
        F.col("is_good_service")
        ==
        F.col("is_disrupted")
    )
)

if invalid_flags_df.count() > 0:
    display(
        invalid_flags_df
    )

    raise ValueError(
        "Invalid service-state flags detected."
    )

print(
    "Service-state validation passed."
)

In [0]:
serving_df = (
    current_status_df
    .withColumn(
        "serving_updated_at",
        F.current_timestamp(),
    )
)

In [0]:
(
    serving_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Created serving table: "
    f"{TARGET_TABLE}"
)

In [0]:
%sql
SELECT
    line_name,
    status_description,
    status_reason,
    is_good_service,
    is_disrupted,
    status_snapshot_at_local
FROM workspace.urbanpulse_gold.current_line_status
ORDER BY
    is_disrupted DESC,
    line_name;

In [0]:
%sql
SELECT
    line_id,
    COUNT(*) AS records
FROM workspace.urbanpulse_gold.current_line_status
GROUP BY line_id
HAVING COUNT(*) <> 1;

In [0]:
%sql
SELECT
    l.line_id,
    l.line_name
FROM workspace.urbanpulse_gold.dim_line l

LEFT ANTI JOIN
workspace.urbanpulse_gold.current_line_status s
    ON l.line_key = s.line_key

WHERE l.is_current = TRUE;

In [0]:
%sql
SELECT s.*
FROM workspace.urbanpulse_gold.current_line_status s

INNER JOIN workspace.urbanpulse_gold.dim_line l
    ON s.line_key = l.line_key

WHERE l.is_current = FALSE;

In [0]:
%sql
SELECT
    COUNT(*) AS total_lines,

    SUM(
        CASE
            WHEN is_good_service THEN 1
            ELSE 0
        END
    ) AS good_service_lines,

    SUM(
        CASE
            WHEN is_disrupted THEN 1
            ELSE 0
        END
    ) AS disrupted_lines,

    ROUND(
        100.0
        *
        SUM(
            CASE
                WHEN is_good_service THEN 1
                ELSE 0
            END
        )
        /
        COUNT(*),
        1
    ) AS good_service_pct

FROM workspace.urbanpulse_gold.current_line_status;

In [0]:
%sql
SELECT
    line_name,
    status_description,
    status_reason,
    status_severity,
    status_snapshot_at_local
FROM workspace.urbanpulse_gold.current_line_status
WHERE is_disrupted = TRUE
ORDER BY
    status_severity,
    line_name;

In [0]:
%sql
SELECT
    MIN(status_snapshot_at_utc) AS oldest_status,
    MAX(status_snapshot_at_utc) AS newest_status,
    MAX(serving_updated_at) AS serving_updated_at
FROM workspace.urbanpulse_gold.current_line_status;

In [0]:
%sql
SELECT COUNT(*) AS rows
FROM workspace.urbanpulse_gold.current_line_status;